# Hisse Fiyat Tahmini: RNN, LSTM, GRU

Log-return tabanlı time-series regresyonu. Üç model aynı veride eğitilip RMSE, MAE, MAPE ve yön doğruluğu ile karşılaştırılıyor.

In [ ]:
import os, sys, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)

from src.data import fetch_prices, build_dataset
from src.model import make_model
from src.engine import train_model, metrics, predict_price, directional_accuracy, set_seed
from src.forecast import recursive_forecast

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## Konfigürasyon

In [ ]:
CFG = dict(ticker="AMZN", start="2010-01-01", end="2023-01-01",
           lookback=20, hidden=32, layers=2, epochs=300, lr=0.01, patience=25)
CFG

## Veri ve özellikler

In [ ]:
df_raw = fetch_prices(CFG["ticker"], CFG["start"], CFG["end"])
data = build_dataset(df_raw, lookback=CFG["lookback"])
print(data["features"])
data["df"][["Close", "ret", "close_sma10", "close_sma50", "rsi"]].tail()

In [ ]:
d = data["df"]
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True,
                       gridspec_kw={"height_ratios": [3, 1, 1]})
ax[0].plot(d.index, d["Close"]); ax[0].set_title(f"{CFG['ticker']} Close"); ax[0].grid(alpha=.3)
ax[1].plot(d.index, d["ret"], color="teal", lw=.6); ax[1].axhline(0, c="k", lw=.5)
ax[1].set_ylabel("log-return"); ax[1].grid(alpha=.3)
ax[2].plot(d.index, d["rsi"]*100, color="purple", lw=.8)
ax[2].axhline(70, ls="--", c="r", alpha=.5); ax[2].axhline(30, ls="--", c="g", alpha=.5)
ax[2].set_ylabel("RSI"); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Eğitim

In [ ]:
os.makedirs("../outputs/models", exist_ok=True)
os.makedirs("../outputs/metrics", exist_ok=True)
os.makedirs("../outputs/figures", exist_ok=True)

y_true = data["yprice_te"]
models, hist, results, preds = {}, {}, {}, {}

set_seed(42)
for kind in ["rnn", "lstm", "gru"]:
    print(f"{kind.upper()} egitiliyor...")
    m = make_model(kind, input_dim=len(data["features"]), hidden=CFG["hidden"], layers=CFG["layers"])
    m, h = train_model(m, data, epochs=CFG["epochs"], lr=CFG["lr"], patience=CFG["patience"], device=device)
    p = predict_price(m, data["X_te"], data["anchor_te"], data["rscaler"], device)
    r = metrics(y_true, p)
    r["DirAcc_%"] = round(directional_accuracy(y_true, p, data["anchor_te"]), 2)
    r["Train_s"] = round(h["time"], 2)
    models[kind], hist[kind], preds[kind], results[kind.upper()] = m, h, p, r
    print(r)

## Değerlendirme

In [ ]:
naive = data["anchor_te"]
results["NAIVE"] = {**metrics(y_true, naive), "DirAcc_%": None, "Train_s": 0.0}
res_df = pd.DataFrame(results).T
res_df.to_csv("../outputs/metrics/comparison.csv")
res_df

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for a, kind in zip(ax, ["rnn", "lstm", "gru"]):
    a.plot(y_true.ravel(), label="Actual", lw=2)
    a.plot(preds[kind].ravel(), label=f"{kind.upper()} pred", alpha=.8)
    a.set_title(f"{kind.upper()} RMSE={results[kind.upper()]['RMSE']:.2f}")
    a.legend(); a.grid(alpha=.3)
plt.suptitle(f"{CFG['ticker']} test seti")
plt.tight_layout()
plt.savefig("../outputs/figures/predictions.png", dpi=120, bbox_inches="tight"); plt.show()

## Forecast

In [ ]:
best = res_df.drop("NAIVE")["RMSE"].astype(float).idxmin().lower()
future = recursive_forecast(models[best], data, CFG["lookback"], steps=30, device=device)

hist_close = data["df"]["Close"].values[-120:]
plt.figure(figsize=(11, 4))
plt.plot(range(len(hist_close)), hist_close, label="History")
plt.plot(range(len(hist_close), len(hist_close)+30), future, "--", label=f"{best.upper()} 30 gun")
plt.title(f"{CFG['ticker']} {best.upper()} forecast")
plt.legend(); plt.grid(alpha=.3)
plt.savefig("../outputs/figures/forecast.png", dpi=120, bbox_inches="tight"); plt.show()

## Kayıt

In [ ]:
for kind, m in models.items():
    torch.save(m.state_dict(), f"../outputs/models/{kind}.pt")
with open("../outputs/models/scaler.pkl", "wb") as f:
    pickle.dump({"fscaler": data["fscaler"], "rscaler": data["rscaler"]}, f)
with open("../outputs/metrics/metrics.json", "w") as f:
    json.dump(results, f, indent=2)
with open("../outputs/models/config.json", "w") as f:
    json.dump(CFG, f, indent=2)
print("kaydedildi")

## Sonuç

Modeller naive baseline ile aynı seviyede RMSE veriyor. Günlük getiri tahmini zor olduğu için bu beklenen bir sonuç. Yön doğruluğu %50 civarındaysa model şans seviyesinde, %53 üzerindeyse hafif bir sinyal var demektir.